# Pre-processing FinChat corpus for use in CE analysis

In [1]:
import os
import sys
import pandas as pd
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

In [31]:
data_path = '../data'
chas_path = os.path.join(data_path, 'finchat-src')
outputs_path = os.path.join(data_path, 'server_ready', 'xling-finchat_corpus.parquet')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [4]:
all_files = grab_all_files(chas_path, 'conversations.csv')
len(all_files), all_files[0]

(1, '../data/finchat-src/data/finchat_chat_conversations.csv')

In [5]:
data = []
for f in all_files:
        
        data += [pd.read_csv(f)]

data = pd.concat(data, ignore_index=True)

In [6]:
data.shape

(3628, 4)

In [7]:
data.head()

,CHAT_ID,SPEAKER_ID,TIME,TEXT
0,4,3,11:10:22.187,Moi
1,4,7,11:10:27.521,Tere!
2,4,3,11:10:46.830,Ootko nähnyt jotain hyviä leffoja viimeaikoina?
3,4,7,11:11:48.573,"Viimeksi taisin käydä keväällä, kun piti kulut..."
4,4,3,11:12:31.667,"Hmm en oo kai kuullukaan tosta, millane leffa ..."


In [8]:
data = data.rename(columns={'CHAT_ID': 'conversation_id', 'SPEAKER_ID': 'speaker', 'TEXT': 'text'})

In [9]:
data['conversation_id'].nunique()

85

In [10]:
data['corpus'] = 'finchat-fin'

In [11]:
data['conversation_id'] = data['corpus'] + '-' + data['conversation_id'].astype(str)

In [13]:
data['utterance_no'] = 0
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    data['utterance_no'].loc[sub.index] = list(range(len(sub)))

100%|██████████| 85/85 [00:00<00:00, 2885.46it/s]


In [14]:
data.head()

,conversation_id,speaker,TIME,text,corpus,utterance_no
0,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0
1,finchat-fin-4,7,11:10:27.521,Tere!,finchat-fin,1
2,finchat-fin-4,3,11:10:46.830,Ootko nähnyt jotain hyviä leffoja viimeaikoina?,finchat-fin,2
3,finchat-fin-4,7,11:11:48.573,"Viimeksi taisin käydä keväällä, kun piti kulut...",finchat-fin,3
4,finchat-fin-4,3,11:12:31.667,"Hmm en oo kai kuullukaan tosta, millane leffa ...",finchat-fin,4


### Correcting utterances/removing CLAN specific formatting.

In [15]:
import re

In [16]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    
    res = res.replace(':', '')
    res = res.replace('\t', ' ')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    
    return res.strip()

In [17]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

100%|██████████| 3628/3628 [00:00<00:00, 143798.82it/s]


In [18]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,finchat-fin,Moi,Moi
1,finchat-fin,Tere!,Tere
2,finchat-fin,Ootko nähnyt jotain hyviä leffoja viimeaikoina?,Ootko nähnyt jotain hyviä leffoja viimeaikoina
3,finchat-fin,"Viimeksi taisin käydä keväällä, kun piti kulut...",Viimeksi taisin käydä keväällä kun piti kulutt...
4,finchat-fin,"Hmm en oo kai kuullukaan tosta, millane leffa ...",Hmm en oo kai kuullukaan tosta millane leffa s...
5,finchat-fin,"Se oli japanilainen draama, missä tämmönen näp...",Se oli japanilainen draama missä tämmönen näpi...


## Create juxtaposed corpus: (x,y) pairs

In [19]:
max_turns_apart = 200

In [20]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

100%|██████████| 85/85 [00:02<00:00, 33.61it/s] 


In [21]:
data = pd.DataFrame(corpus)
print(data.shape)
data.head()

(100099, 11)


,conversation_id,speaker,TIME,text,corpus,utterance_no,raw_text,next_speaker,next_text,next_utterance_no,next_utterance_delta_no
0,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Tere,1,0
1,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Viimeksi taisin käydä keväällä kun piti kulutt...,3,1
2,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se oli japanilainen draama missä tämmönen näpi...,5,2
3,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se voitti kultaisen palmun tai jotain,7,3
4,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,En saa millään mieleen ohjaajan nimeä,8,4


In [21]:
# del data['raw_text']

In [22]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [23]:
data = data.rename(columns=rename_columns)
data.head()

,conversation_id,speaker,TIME,x_utterance,corpus,utterance_no,raw_text,y_speaker,y_utterance,y_utterance_no,y_utterance_delta_no
0,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Tere,1,0
1,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Viimeksi taisin käydä keväällä kun piti kulutt...,3,1
2,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se oli japanilainen draama missä tämmönen näpi...,5,2
3,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se voitti kultaisen palmun tai jotain,7,3
4,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,En saa millään mieleen ohjaajan nimeä,8,4


In [24]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

,conversation_id,x_speaker,TIME,x_utterance,corpus,x_turn_id,raw_text,y_speaker,y_utterance,y_turn_id,y_utterance_delta_no
0,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Tere,1,0
1,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Viimeksi taisin käydä keväällä kun piti kulutt...,3,1
2,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se oli japanilainen draama missä tämmönen näpi...,5,2
3,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,Se voitti kultaisen palmun tai jotain,7,3
4,finchat-fin-4,3,11:10:22.187,Moi,finchat-fin,0,Moi,7,En saa millään mieleen ohjaajan nimeä,8,4


In [25]:
data['x_speaker'] = data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [26]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [27]:
data['self'].value_counts()

self
True     50218
False    49881
Name: count, dtype: int64

In [28]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [29]:
# # corpus sanity check . . . make sure that conversation_ids are all unique 
# k = data[['conversation_id', 'corpus']].drop_duplicates()
# k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

In [29]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,finchat-fin,Seuraatko jotain urheilua,finchat-fin-10-0,finchat-fin-10-5,Melko harvoin ehkä futista joskus Entä itse,0,1
1,finchat-fin,Seuraatko jotain urheilua,finchat-fin-10-0,finchat-fin-10-5,Jalkapallossa vai missä tahansa lajissa,0,3
2,finchat-fin,Seuraatko jotain urheilua,finchat-fin-10-0,finchat-fin-10-0,Enpä juuri Maaotteluita ehkä,0,2
3,finchat-fin,Seuraatko jotain urheilua,finchat-fin-10-0,finchat-fin-10-0,Lätkä lähinnä Joskus myös muitakin,0,4
4,finchat-fin,Melko harvoin ehkä futista joskus Entä itse,finchat-fin-10-5,finchat-fin-10-0,Enpä juuri Maaotteluita ehkä,1,2


In [30]:
data.shape

(100099, 12)

## Save outputs for server operations.

In [32]:
# data.to_csv(outputs_path, index=False, encoding='utf-8', sep='\t')
data.to_parquet(
    outputs_path,
    engine='fastparquet',
    compression='snappy'
)